# **BRIGHT — HFExtractor Demo & Ablation Evaluation**
<a target="_blank" href="https://colab.research.google.com/github/raphaelrubrice/BRIGHT_ClinicalDataBase/blob/final/notebooks/run_hf_demo.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

This notebook:
1. Sets up the Colab environment
2. Smoke-tests `HFExtractor` in isolation
3. Runs the full 3-mode ablation (`rule` / `ml` / `both`) via `run_pipeline_ablation.py`
4. Evaluates results against gold-standard annotations (exact + relaxed F1)
5. Produces professional seaborn plots per pipeline version and a cross-version comparison

## §1 — Colab setup

In [1]:
from google.colab import drive
drive.mount('/content/drive/')

%cd /content/drive/MyDrive/MVA/BRIGHT

Mounted at /content/drive/
/content/drive/MyDrive/MVA/BRIGHT


### **/!\ Following cell removes the previous repo clone**

In [2]:
!rm -rf BRIGHT_ClinicalDataBase

### Clone & checkout

In [3]:
!git clone https://github.com/raphaelrubrice/BRIGHT_ClinicalDataBase.git
%cd BRIGHT_ClinicalDataBase
!git checkout final

Cloning into 'BRIGHT_ClinicalDataBase'...
remote: Enumerating objects: 996, done.
remote: Counting objects: 100% (410/410), done.
remote: Compressing objects: 100% (305/305), done.
remote: Total 996 (delta 147), reused 338 (delta 88), pack-reused 586 (from 1)
Receiving objects: 100% (996/996), 1.03 MiB | 4.37 MiB/s, done.
Resolving deltas: 100% (479/479), done.
/content/drive/MyDrive/MVA/BRIGHT/BRIGHT_ClinicalDataBase
Updating files: 100% (99/99), done.
Branch 'final' set up to track remote branch 'final' from 'origin'.
Switched to a new branch 'final'


### Copy data from Drive

In [4]:
!cp -r ../data/ .
!cp -r ../test_annotated/ .

### Install dependencies

In [ ]:
!pip install -q torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1

# edsnlp pulls in a compatible spaCy — install it before requirements.txt so
# the pinned version isn't overwritten by the bare `spacy` line there.
!pip install -q "edsnlp[ml]>=0.17.0"

# Fix known markupsafe / numpy ABI conflicts (mirrors setup.sh line 3)
!pip install -q --force-reinstall markupsafe jinja2 "numpy<2.0" pillow

# Remaining project deps (omit edsnlp / spaCy — already pinned above)
!pip install -q pymupdf portalocker "pydantic>=2.0" rapidfuzz pandas \
    matplotlib seaborn langdetect onnxruntime \
    "gliner2==1.2.4" gliner2-onnx huggingface_hub transformers datasets

# Download the French spaCy model AFTER all packages are stable
!python -m spacy download fr_core_news_lg -q

print("✓ All packages installed.")
print("↻ Restarting kernel so Cython extensions load into a clean process …")
print("   Colab will reconnect automatically — re-run from the next cell (§HF Login).")

import os
os.kill(os.getpid(), 9)   # SIGKILL → Colab auto-reconnects; no data is lost

### §1b — HuggingFace login  *(run this after the kernel restarts)*

The runtime was restarted by the previous cell.  
**Start re-running from here** — all packages are already installed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive/', force_remount=False)   # no-op if already mounted

# Re-enter the project directory (needed after kernel restart)
import os
os.chdir('/content/drive/MyDrive/MVA/BRIGHT/BRIGHT_ClinicalDataBase')
print("CWD:", os.getcwd())

# Authenticate with HuggingFace (shows a secure token-input widget)
from huggingface_hub import notebook_login
notebook_login()

---
## §2 — HFExtractor smoke test

Test `HFExtractor` in isolation with 2 groups on a single example sentence.

In [10]:
import sys
sys.path.insert(0, '.')

from src.extraction.hf_extractor import HFExtractor

hf = HFExtractor(enabled_groups=["diagnosis", "tumor_location"])

example = "Glioblastome WHO grade IV, temporal gauche, IDH1 non muté."
result = hf.extract(example)

print(f"Extracted {len(result)} fields from example sentence:\n")
for field, ev in sorted(result.items()):
    print(f"  {field:<32} = {ev.value!r:20}  (span: {ev.source_span!r})")

KeyError: '__reduce_cython__'

---
## §3 — Full ablation run (3 modes)

Each mode writes a `results_<mode>_<timestamp>.json` to `/tmp/bright_eval/<mode>/`.

In [ ]:
import subprocess
import sys

EVAL_DIR = "/tmp/bright_eval"
MAX_DOCS = 50   # set to None to run on all gold-standard docs

for mode in ["rule", "ml", "both"]:
    print(f"\n{'='*60}\nRunning mode: {mode}\n{'='*60}", flush=True)
    cmd = [
        sys.executable, "scripts/run_pipeline_ablation.py",
        "--mode", mode,
        "--out-dir", f"{EVAL_DIR}/{mode}",
        "--jobs", "1",
    ]
    if MAX_DOCS is not None:
        cmd += ["--max-docs", str(MAX_DOCS)]
    proc = subprocess.run(cmd, check=False)
    if proc.returncode != 0:
        print(f"[WARNING] mode={mode} exited with code {proc.returncode}")

---
## §4 — Load results & gold standard

In [ ]:
import json
import glob
from pathlib import Path

def load_latest_results(mode_dir: str) -> list[dict]:
    """Load the most recent results JSON written by run_pipeline_ablation."""
    files = sorted(glob.glob(f"{mode_dir}/results_*.json"))
    if not files:
        raise FileNotFoundError(f"No results_*.json in {mode_dir}")
    with open(files[-1], encoding="utf-8") as f:
        return json.load(f)

results_by_mode: dict[str, list[dict]] = {}
for mode in ["rule", "ml", "both"]:
    results_by_mode[mode] = load_latest_results(f"{EVAL_DIR}/{mode}")
    print(f"[{mode}] {len(results_by_mode[mode])} documents loaded")

# Load gold-standard annotations
gold_dir = Path("data/gold_standard")
gold_by_id: dict[str, dict] = {}
for p in sorted(gold_dir.glob("*.json")):
    if p.name == "manifest.json":
        continue
    d = json.loads(p.read_text(encoding="utf-8"))
    gold_by_id[d["document_id"]] = d.get("annotations", {})
print(f"\nGold standard: {len(gold_by_id)} documents")

---
## §5 — Metrics computation

Two regimes:
- **Exact**: `str(pred).lower().strip() == str(gold).lower().strip()`
- **Relaxed**: fuzzy `rapidfuzz.fuzz.ratio ≥ 80` for free-text fields; exact for controlled fields

In [ ]:
from rapidfuzz import fuzz
import pandas as pd
from collections import defaultdict

# Free-text fields where fuzzy matching is appropriate
FREE_TEXT_FIELDS = {
    "diag_histologique", "diag_integre", "chimios", "chimio_protocole",
    "neuroncologue", "neurochirurgien", "radiotherapeute", "anatomo_pathologiste",
    "ik_clinique", "qualite_exerese", "evol_clinique", "survie_globale",
    "autre_trouble_1er_symptome", "localisation_chir", "tumeur_position",
    "localisation_radiotherapie",
    # Date fields
    "date_chir", "date_rcp", "dn_date", "date_deces", "date_1er_symptome",
    "exam_radio_date_decouverte", "date_progression", "chm_date_debut", "chm_date_fin",
    "rx_date_debut", "rx_date_fin",
}

FUZZY_THRESHOLD = 0.80


def normalise_value(v) -> str:
    return str(v).lower().strip() if v is not None else ""


def is_tp_exact(pred, gold) -> bool:
    return normalise_value(pred) == normalise_value(gold)


def is_tp_relaxed(pred, gold, field: str) -> bool:
    p, g = normalise_value(pred), normalise_value(gold)
    if not p or not g:
        return False
    if field in FREE_TEXT_FIELDS:
        return fuzz.ratio(p, g) / 100 >= FUZZY_THRESHOLD
    return p == g  # controlled fields: exact only


def compute_prf(tp: int, fp: int, fn: int) -> dict:
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    return {"precision": prec, "recall": rec, "f1": f1, "tp": tp, "fp": fp, "fn": fn}


def evaluate_mode(results: list[dict], gold_by_id: dict) -> dict:
    """
    Returns per_field_metrics: {field: {"exact": {...}, "relaxed": {...}}}
    """
    exact_counters  = defaultdict(lambda: {"tp": 0, "fp": 0, "fn": 0})
    relaxed_counters = defaultdict(lambda: {"tp": 0, "fp": 0, "fn": 0})

    for rec in results:
        doc_id = rec["document_id"]
        gold = gold_by_id.get(doc_id, {})
        preds = rec.get("features", {})

        all_fields = set(gold.keys()) | set(preds.keys())
        for field in all_fields:
            gold_val = gold.get(field, {}).get("value") if isinstance(gold.get(field), dict) else gold.get(field)
            pred_ev  = preds.get(field)
            pred_val = pred_ev.get("value") if isinstance(pred_ev, dict) else None

            has_gold = gold_val is not None and normalise_value(gold_val) != ""
            has_pred = pred_val is not None and normalise_value(pred_val) != ""

            # --- Exact ---
            if has_gold and has_pred:
                if is_tp_exact(pred_val, gold_val):
                    exact_counters[field]["tp"] += 1
                else:
                    exact_counters[field]["fp"] += 1
                    exact_counters[field]["fn"] += 1
            elif has_pred and not has_gold:
                exact_counters[field]["fp"] += 1
            elif has_gold and not has_pred:
                exact_counters[field]["fn"] += 1

            # --- Relaxed ---
            if has_gold and has_pred:
                if is_tp_relaxed(pred_val, gold_val, field):
                    relaxed_counters[field]["tp"] += 1
                else:
                    relaxed_counters[field]["fp"] += 1
                    relaxed_counters[field]["fn"] += 1
            elif has_pred and not has_gold:
                relaxed_counters[field]["fp"] += 1
            elif has_gold and not has_pred:
                relaxed_counters[field]["fn"] += 1

    per_field: dict[str, dict] = {}
    all_fields = set(exact_counters) | set(relaxed_counters)
    for field in all_fields:
        ec = exact_counters[field]
        rc = relaxed_counters[field]
        per_field[field] = {
            "exact":   compute_prf(ec["tp"], ec["fp"], ec["fn"]),
            "relaxed": compute_prf(rc["tp"], rc["fp"], rc["fn"]),
        }
    return per_field


metrics_by_mode: dict[str, dict] = {}
for mode in ["rule", "ml", "both"]:
    metrics_by_mode[mode] = evaluate_mode(results_by_mode[mode], gold_by_id)
    n_fields = len(metrics_by_mode[mode])
    macro_f1_exact   = sum(v["exact"]["f1"]   for v in metrics_by_mode[mode].values()) / max(n_fields, 1)
    macro_f1_relaxed = sum(v["relaxed"]["f1"] for v in metrics_by_mode[mode].values()) / max(n_fields, 1)
    print(f"[{mode}] {n_fields} fields | macro-F1 exact={macro_f1_exact:.3f} relaxed={macro_f1_relaxed:.3f}")

---
## §6 — Seaborn plots

Saved to `/tmp/bright_eval/plots/{mode}/` and `/tmp/bright_eval/plots/comparison/`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

sns.set_theme(style="whitegrid", font_scale=0.9)
PLOT_ROOT = Path("/tmp/bright_eval/plots")
TS = datetime.now().strftime("%Y%m%d_%H%M")

# Color palettes
PAL_PRF = sns.color_palette("Set2", 3)   # Precision / Recall / F1
PAL_MODES = sns.color_palette("deep", 3) # rule / ml / both
MODE_COLORS = {"rule": PAL_MODES[0], "ml": PAL_MODES[1], "both": PAL_MODES[2]}


def plot_per_field(mode: str, regime: str, per_field: dict) -> None:
    """Horizontal bar chart of P/R/F1 per field, sorted by F1 descending."""
    rows = []
    for field, vals in per_field.items():
        m = vals[regime]
        rows.append({"field": field, "Precision": m["precision"],
                     "Recall": m["recall"], "F1": m["f1"]})
    df = pd.DataFrame(rows).sort_values("F1", ascending=True)

    n_fields = len(df)
    fig, ax = plt.subplots(figsize=(14, max(8, n_fields * 0.30)))

    bar_h = 0.25
    y = range(n_fields)
    for i, (metric, color) in enumerate(zip(
            ["Precision", "Recall", "F1"], PAL_PRF)):
        bars = ax.barh(
            [yi + (i - 1) * bar_h for yi in y],
            df[metric], bar_h, label=metric, color=color, alpha=0.85
        )
        for bar in bars:
            w = bar.get_width()
            if w >= 0.01:
                ax.text(w + 0.005, bar.get_y() + bar.get_height() / 2,
                        f"{w:.2f}", va="center", fontsize=6)

    ax.set_yticks(list(y))
    ax.set_yticklabels(df["field"].tolist(), fontsize=8)
    ax.set_xlim(0, 1.12)
    ax.set_xlabel("Score")
    ax.set_title(f"[{mode}] — Per-field {regime.capitalize()} P / R / F1", fontsize=12)
    ax.legend(loc="lower right")
    ax.grid(axis="x", alpha=0.4)
    plt.tight_layout()

    out_dir = PLOT_ROOT / mode
    out_dir.mkdir(parents=True, exist_ok=True)
    fname = f"{mode}__{regime}__per_field__{TS}.png"
    fig.savefig(out_dir / fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out_dir / fname}")


# Generate per-mode per-field plots
for mode in ["rule", "ml", "both"]:
    for regime in ["exact", "relaxed"]:
        plot_per_field(mode, regime, metrics_by_mode[mode])

In [ ]:
def plot_overall_comparison(regime: str) -> None:
    """Grouped bar chart: rule / ml / both × Precision / Recall / F1-macro."""
    rows = []
    for mode in ["rule", "ml", "both"]:
        pf = metrics_by_mode[mode]
        n = max(len(pf), 1)
        macro_p  = sum(v[regime]["precision"] for v in pf.values()) / n
        macro_r  = sum(v[regime]["recall"]    for v in pf.values()) / n
        macro_f1 = sum(v[regime]["f1"]        for v in pf.values()) / n
        rows.append({"mode": mode, "Precision": macro_p,
                     "Recall": macro_r, "F1-macro": macro_f1})

    df = pd.DataFrame(rows).set_index("mode")
    metrics = ["Precision", "Recall", "F1-macro"]

    x = range(len(metrics))
    bar_w = 0.25
    fig, ax = plt.subplots(figsize=(10, 5))
    for i, (mode, color) in enumerate(MODE_COLORS.items()):
        vals = [df.loc[mode, m] for m in metrics]
        bars = ax.bar(
            [xi + (i - 1) * bar_w for xi in x],
            vals, bar_w, label=mode.capitalize(), color=color, alpha=0.85
        )
        for bar in bars:
            h = bar.get_height()
            ax.text(bar.get_x() + bar.get_width() / 2, h + 0.005,
                    f"{h:.3f}", ha="center", va="bottom", fontsize=8)

    ax.set_xticks(list(x))
    ax.set_xticklabels(metrics)
    ax.set_ylim(0, 1.1)
    ax.set_ylabel("Score (macro-average)")
    ax.set_title(f"Pipeline Comparison — {regime.capitalize()} Macro P / R / F1", fontsize=13)
    ax.legend(title="Pipeline")
    ax.grid(axis="y", alpha=0.4)
    plt.tight_layout()

    out_dir = PLOT_ROOT / "comparison"
    out_dir.mkdir(parents=True, exist_ok=True)
    fname = f"comparison__{regime}__overall_prf__{TS}.png"
    fig.savefig(out_dir / fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out_dir / fname}")


plot_overall_comparison("exact")
plot_overall_comparison("relaxed")

---
## §7 — Summary table

In [ ]:
summary_rows = []
for mode in ["rule", "ml", "both"]:
    pf = metrics_by_mode[mode]
    n = max(len(pf), 1)
    row = {"mode": mode}
    for regime in ["exact", "relaxed"]:
        row[f"macro_P_{regime}"]  = round(sum(v[regime]["precision"] for v in pf.values()) / n, 4)
        row[f"macro_R_{regime}"]  = round(sum(v[regime]["recall"]    for v in pf.values()) / n, 4)
        row[f"macro_F1_{regime}"] = round(sum(v[regime]["f1"]        for v in pf.values()) / n, 4)
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).set_index("mode")
display(summary_df.style.format("{:.4f}").background_gradient(cmap="YlGn", axis=0))